<a href="https://colab.research.google.com/github/Pixeler5diti/Delhi-High-Court-Virtual-Judge/blob/main/finetuned_legalbert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install --upgrade pymupdf


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 78.1 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os

pdf_folder = "/content/drive/My Drive/judgments/"
pdf_files = [os.path.join(pdf_folder, f) for f in os.listdir(pdf_folder) if f.endswith(".pdf")]

print("Found PDFs:")
for f in pdf_files:
    print(f)


Found PDFs:
/content/drive/My Drive/judgments/1.pdf
/content/drive/My Drive/judgments/2.pdf
/content/drive/My Drive/judgments/3.pdf
/content/drive/My Drive/judgments/4.pdf
/content/drive/My Drive/judgments/5.pdf
/content/drive/My Drive/judgments/6.pdf
/content/drive/My Drive/judgments/7.pdf
/content/drive/My Drive/judgments/8.pdf
/content/drive/My Drive/judgments/9.pdf
/content/drive/My Drive/judgments/10.pdf
/content/drive/My Drive/judgments/11.pdf
/content/drive/My Drive/judgments/12.pdf
/content/drive/My Drive/judgments/13.pdf
/content/drive/My Drive/judgments/20.pdf
/content/drive/My Drive/judgments/24.pdf
/content/drive/My Drive/judgments/21.pdf
/content/drive/My Drive/judgments/23.pdf
/content/drive/My Drive/judgments/19.pdf
/content/drive/My Drive/judgments/17.pdf
/content/drive/My Drive/judgments/22.pdf
/content/drive/My Drive/judgments/14.pdf
/content/drive/My Drive/judgments/16.pdf
/content/drive/My Drive/judgments/18.pdf
/content/drive/My Drive/judgments/15.pdf
/content/driv

In [ ]:
!pip install -q pdfminer.six nltk scikit-learn sentence-transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 60.4 MB/s eta 0:00:00


In [ ]:
!pip install --upgrade pymupdf

In [ ]:
import fitz  # PyMuPDF
import os
import json

def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

def preprocess_text(text):
    # Normalize whitespace and basic cleaning
    lines = text.split('\n')
    lines = [line.strip() for line in lines if line.strip()]
    return "\n".join(lines)

def split_sections(text):
    lower_text = text.lower()
    sections = {
        "title": text.split('\n')[0],
        "facts": "",
        "arguments": "",
        "judgment": ""
    }

    # Best-effort splitting
    if "facts" in lower_text:
        sections["facts"] = text.split("facts", 1)[-1].split("arguments", 1)[0] if "arguments" in lower_text else ""
    if "arguments" in lower_text:
        sections["arguments"] = text.split("arguments", 1)[-1].split("judgment", 1)[0] if "judgment" in lower_text else ""
    if "judgment" in lower_text:
        sections["judgment"] = text.split("judgment", 1)[-1]
    return sections

def process_pdf(pdf_path, output_dir="./extracted_cases"):
    # Ensure the output directory exists
    os.makedirs(output_dir, exist_ok=True)
    # Extract and clean text
    raw_text = extract_text_from_pdf(pdf_path)
    clean_text = preprocess_text(raw_text)
    case_data = split_sections(clean_text)

    # Save JSON
    filename = os.path.splitext(os.path.basename(pdf_path))[0]
    output_path = os.path.join(output_dir, f"{filename}.json")
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(case_data, f, indent=2)

    print(f"[✓] Processed: {filename} → {output_path}")

# Example usage
pdfs = [
    "/content/drive/MyDrive/judgments/1.pdf",
    "/content/drive/MyDrive/judgments/2.pdf",
    "/content/drive/MyDrive/judgments/3.pdf",
    "/content/drive/MyDrive/judgments/4.pdf",
    "/content/drive/MyDrive/judgments/5.pdf",
    "/content/drive/MyDrive/judgments/6.pdf",
    "/content/drive/MyDrive/judgments/7.pdf",
    "/content/drive/MyDrive/judgments/8.pdf",
    "/content/drive/MyDrive/judgments/9.pdf",
    "/content/drive/MyDrive/judgments/10.pdf",
    "/content/drive/MyDrive/judgments/11.pdf",
    "/content/drive/MyDrive/judgments/12.pdf",
    "/content/drive/MyDrive/judgments/13.pdf",
    "/content/drive/MyDrive/judgments/14.pdf",
    "/content/drive/MyDrive/judgments/15.pdf",
    "/content/drive/MyDrive/judgments/16.pdf",
    "/content/drive/MyDrive/judgments/17.pdf",
    "/content/drive/MyDrive/judgments/18.pdf",
    "/content/drive/MyDrive/judgments/19.pdf",
    "/content/drive/MyDrive/judgments/20.pdf",
    "/content/drive/MyDrive/judgments/21.pdf",
    "/content/drive/MyDrive/judgments/22.pdf",
    "/content/drive/MyDrive/judgments/23.pdf",
    "/content/drive/MyDrive/judgments/24.pdf",
    "/content/drive/MyDrive/judgments/25.pdf",
    "/content/drive/MyDrive/judgments/26.pdf",
    "/content/drive/MyDrive/judgments/27.pdf",
    "/content/drive/MyDrive/judgments/28.pdf",
    "/content/drive/MyDrive/judgments/29.pdf",
    "/content/drive/MyDrive/judgments/30.pdf",]


# Define Google Drive output path
output_dir_drive = "/content/drive/MyDrive/processed_json"

# Process all PDFs and save to Google Drive
for pdf in pdfs:
    process_pdf(pdf, output_dir=output_dir_drive)


[✓] Processed: 1 → /content/drive/MyDrive/processed_json/1.json
[✓] Processed: 2 → /content/drive/MyDrive/processed_json/2.json
[✓] Processed: 3 → /content/drive/MyDrive/processed_json/3.json
[✓] Processed: 4 → /content/drive/MyDrive/processed_json/4.json
[✓] Processed: 5 → /content/drive/MyDrive/processed_json/5.json
[✓] Processed: 6 → /content/drive/MyDrive/processed_json/6.json
[✓] Processed: 7 → /content/drive/MyDrive/processed_json/7.json
[✓] Processed: 8 → /content/drive/MyDrive/processed_json/8.json
[✓] Processed: 9 → /content/drive/MyDrive/processed_json/9.json
[✓] Processed: 10 → /content/drive/MyDrive/processed_json/10.json
[✓] Processed: 11 → /content/drive/MyDrive/processed_json/11.json
[✓] Processed: 12 → /content/drive/MyDrive/processed_json/12.json
[✓] Processed: 13 → /content/drive/MyDrive/processed_json/13.json
[✓] Processed: 14 → /content/drive/MyDrive/processed_json/14.json
[✓] Processed: 15 → /content/drive/MyDrive/processed_json/15.json
[✓] Processed: 16 → /content

In [ ]:
import fitz  # PyMuPDF
import os
import json

def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

def preprocess_text(text):
    lines = text.split('\n')
    lines = [line.strip() for line in lines if line.strip()]
    return "\n".join(lines)

def split_sections(text):
    lower_text = text.lower()
    sections = {
        "title": text.split('\n')[0],
        "facts": "",
        "arguments": "",
        "judgment": ""
    }

    if "facts" in lower_text:
        sections["facts"] = text.split("facts", 1)[-1].split("arguments", 1)[0] if "arguments" in lower_text else ""
    if "arguments" in lower_text:
        sections["arguments"] = text.split("arguments", 1)[-1].split("judgment", 1)[0] if "judgment" in lower_text else ""
    if "judgment" in lower_text:
        sections["judgment"] = text.split("judgment", 1)[-1]
    return sections

def process_all_pdfs_to_single_json(pdf_paths, output_file_path):
    all_cases = []
    for pdf_path in pdf_paths:
        raw_text = extract_text_from_pdf(pdf_path)
        clean_text = preprocess_text(raw_text)
        case_data = split_sections(clean_text)
        all_cases.append(case_data)

    with open(output_file_path, "w", encoding="utf-8") as f:
        json.dump(all_cases, f, indent=2)
    print(f"[✓] Dataset saved to {output_file_path}")

# Your PDF paths
pdfs = [f"/content/drive/MyDrive/judgments/{i}.pdf" for i in range(1, 31)]

# Output one single dataset-friendly JSON file
output_file = "/content/drive/MyDrive/processed_json/dataset.json"

# Process all PDFs
process_all_pdfs_to_single_json(pdfs, output_file)


[✓] Dataset saved to /content/drive/MyDrive/processed_json/dataset.json


In [ ]:
import json
import pandas as pd

def load_cases_from_json(json_file_path):
    with open(json_file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return pd.DataFrame(data)


In [ ]:
df = load_cases_from_json("/content/drive/MyDrive/processed_json/dataset.json")
print(df.head())


            title                                              facts  \
0               1  , and since the respondent did not raise any s...   
1      REPORTABLE   of\nevery case.”\n5.\nThereafter, vide the or...   
2  2023 INSC 1066                                                      
3   2024 INSC 856   would show that the\ncorrectness of the quest...   
4   2024 INSC 857   of this case, the judgment of the\nHigh Court...   

                                           arguments  \
0   on both sides, finally upheld the plea of the...   
1   on the following issues:\n“1. The scope and e...   
2                                                      
3   requires reassessing Parliament's\nreasoning ...   
4  ; (ii) equal\nopportunities to parties to pres...   

                                            judgment  
0  s and certain International Covenants,\nopined...  
1  s of this Court on the\napplicability of the p...  
2   in Hyundai Engg. case [United India\nInsuranc...  
3   of thi

In [ ]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 17.7 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.


In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="/content/drive/MyDrive/processed_json/dataset.json")


Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
print(dataset)
print(dataset['train'][0])  # Shows first judgment


DatasetDict({
    train: Dataset({
        features: ['title', 'facts', 'arguments', 'judgment'],
        num_rows: 30
    })
})
{'title': '1', 'facts': ', and since the respondent did not raise any such Eighth\nAmendment issue, Justice Powell concurred with the majority.\n32.\nThe dissenting opinion of four Justices makes interesting\nreading. Justice Blackmun, who spoke for four dissenters,\nbegan with the classical definition of the old privacy right which\nis the “right to be let alone”, and quoted from Justice Holmes’\narticle The Path of the Law, stating:-\n“[i]t is revolting to have no better reason for a rule of\nlaw than that so it was laid down in the time of\nHenry IV. It is still more revolting if the grounds upon\nwhich it was laid down have vanished long since,\n23\nand the rule simply persists from blind imitation of\nthe past.”\n33.\nSo much, then, for history and its “ancient roots”. Justice\nBlackmun’s dissent then went on to consider the famous\njudgment in Wisconsin

In [ ]:

df = load_cases_from_json("/content/drive/MyDrive/processed_json/dataset.json")


In [ ]:
!pip install datasets


In [ ]:
from datasets import load_dataset

# Path to your dataset.json
json_path = "/content/drive/MyDrive/processed_json/dataset.json"

# Try loading the dataset
dataset = load_dataset("json", data_files=json_path, split="train")

# Show the number of samples
print(f"✅ Successfully loaded {len(dataset)} samples.")

# Display a sample
print(dataset[0])


Generating train split: 0 examples [00:00, ? examples/s]

✅ Successfully loaded 30 samples.
{'title': '1', 'facts': ', and since the respondent did not raise any such Eighth\nAmendment issue, Justice Powell concurred with the majority.\n32.\nThe dissenting opinion of four Justices makes interesting\nreading. Justice Blackmun, who spoke for four dissenters,\nbegan with the classical definition of the old privacy right which\nis the “right to be let alone”, and quoted from Justice Holmes’\narticle The Path of the Law, stating:-\n“[i]t is revolting to have no better reason for a rule of\nlaw than that so it was laid down in the time of\nHenry IV. It is still more revolting if the grounds upon\nwhich it was laid down have vanished long since,\n23\nand the rule simply persists from blind imitation of\nthe past.”\n33.\nSo much, then, for history and its “ancient roots”. Justice\nBlackmun’s dissent then went on to consider the famous\njudgment in Wisconsin v. Yoder, 32 L. Ed. 2d 15 (1972), in\nwhich the Court had upheld the fundamental right of the 

In [ ]:
pip install transformers datasets torch


In [ ]:
!pip install transformers datasets sentence-transformers scikit-learn pandas torch faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 20.9 MB/s eta 0:00:00


In [ ]:
!pip install --upgrade transformers


In [ ]:
!pip install transformers datasets torch

In [ ]:
import json
import pandas as pd

# Load your dataset
with open("/content/drive/MyDrive/processed_json/dataset.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Convert to DataFrame
df = pd.DataFrame(data)

# Merge all sections into one string for embedding
df["full_text"] = df["title"] + "\n" + df["facts"] + "\n" + df["arguments"] + "\n" + df["judgment"]
df = df[df["judgment"].str.strip() != ""]  # remove empty judgments if any


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import torch
from transformers import BertTokenizer, BertForSequenceClassification
import pandas as pd

#Step 1: Check if GPU is available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Step 2: Load the model and move it to the device
model_cls = BertForSequenceClassification.from_pretrained("nlpaueb/legal-bert-base-uncased", num_labels=2)
model_cls = model_cls.to(device)  # Move the model to the device

# Step 3: Load the tokenizer
tokenizer = BertTokenizer.from_pretrained("nlpaueb/legal-bert-base-uncased")


# Legal-specific model
model = SentenceTransformer("nlpaueb/legal-bert-base-uncased")

# Generate embeddings
embeddings = model.encode(df["full_text"].tolist(), show_progress_bar=True)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at nlpaueb/legal-bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/222k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
!pip install datasets

In [ ]:
import json
import pandas as pd
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import torch

# Load your dataset
with open("/content/drive/MyDrive/processed_json/dataset.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Convert to DataFrame
df = pd.DataFrame(data)

# Merge all sections into one string for embedding
df["full_text"] = df["title"] + "\n" + df["facts"] + "\n" + df["arguments"] + "\n" + df["judgment"]
df = df[df["judgment"].str.strip() != ""]  # remove empty judgments if any

# Create labels for "verdict" — 0 = dismiss, 1 = allow, etc.
df["label"] = df["judgment"].apply(lambda x: 1 if "allowed" in x.lower() else 0)

# 1. Load and customize tokenizer FIRST
tokenizer = BertTokenizer.from_pretrained("nlpaueb/legal-bert-base-uncased")
legal_phrases = ["Article 142", "Section 482", "CrPC", "IPC", "Constitution of India"]  # Add more as needed
tokenizer.add_tokens(legal_phrases)

# 2. THEN, Load the model and resize embeddings
model_cls = BertForSequenceClassification.from_pretrained("nlpaueb/legal-bert-base-uncased", num_labels=2, ignore_mismatched_sizes=True)
model_cls.resize_token_embeddings(len(tokenizer))  # Now model_cls is defined


def tokenize(example):
    return tokenizer(example["judgment"], truncation=True, padding=True)

dataset = Dataset.from_pandas(df[["judgment", "label"]])
dataset = dataset.map(tokenize, batched=True)
dataset = dataset.rename_column("label", "labels")
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# Split dataset
train_test = dataset.train_test_split(test_size=0.2)
train_set = train_test["train"]
test_set = train_test["test"]


training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,  # number of epochs
    per_device_train_batch_size=8,  # batch size per device
    per_device_eval_batch_size=8,   # batch size for evaluation
    warmup_steps=500,  # number of warmup steps
    weight_decay=0.01,  # strength of weight decay
    logging_dir="./logs",  # directory for storing logs
    logging_steps=10,  # logging every 10 steps
      # Check this depending on version
)

trainer = Trainer(
    model=model_cls,
    args=training_args,
    train_dataset=train_set,
    eval_dataset=test_set,
    tokenizer=tokenizer
)

trainer.train()
trainer.evaluate()

In [ ]:
with open("/content/drive/MyDrive/processed_json/dataset.json", 'r') as f:
    sample_data = json.load(f)[:3]  # First 3 entries
print("Sample JSON structure:")
print(sample_data)

Sample JSON structure:
[{'title': '1', 'facts': ', and since the respondent did not raise any such Eighth\nAmendment issue, Justice Powell concurred with the majority.\n32.\nThe dissenting opinion of four Justices makes interesting\nreading. Justice Blackmun, who spoke for four dissenters,\nbegan with the classical definition of the old privacy right which\nis the “right to be let alone”, and quoted from Justice Holmes’\narticle The Path of the Law, stating:-\n“[i]t is revolting to have no better reason for a rule of\nlaw than that so it was laid down in the time of\nHenry IV. It is still more revolting if the grounds upon\nwhich it was laid down have vanished long since,\n23\nand the rule simply persists from blind imitation of\nthe past.”\n33.\nSo much, then, for history and its “ancient roots”. Justice\nBlackmun’s dissent then went on to consider the famous\njudgment in Wisconsin v. Yoder, 32 L. Ed. 2d 15 (1972), in\nwhich the Court had upheld the fundamental right of the Amish\ncom

In [ ]:
df.head()

,title,facts,arguments,judgment,full_text,label
0,1,", and since the respondent did not raise any s...","on both sides, finally upheld the plea of the...","s and certain International Covenants,\nopined...","1\n, and since the respondent did not raise an...",1
1,REPORTABLE,"of\nevery case.”\n5.\nThereafter, vide the or...",on the following issues:\n“1. The scope and e...,s of this Court on the\napplicability of the p...,"REPORTABLE\n of\nevery case.”\n5.\nThereafter,...",1
2,2023 INSC 1066,,,in Hyundai Engg. case [United India\nInsuranc...,2023 INSC 1066\n\n\n in Hyundai Engg. case [Un...,1
3,2024 INSC 856,would show that the\ncorrectness of the quest...,requires reassessing Parliament's\nreasoning ...,of this Court in Azeez\nBasha (supra). They ...,2024 INSC 856\n would show that the\ncorrectne...,1
4,2024 INSC 857,"of this case, the judgment of the\nHigh Court...",; (ii) equal\nopportunities to parties to pres...,of the\nHigh Court cannot be faulted with (si...,"2024 INSC 857\n of this case, the judgment of ...",1


In [ ]:
pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 3.2 MB/s eta 0:00:00


In [ ]:
# 1. FIRST RUN THESE INSTALLATIONS (ONLY ONCE)
!pip install torch transformers datasets evaluate pandas

# 2. THEN RUN THIS COMPLETE SCRIPT
import torch
import json
import pandas as pd
import numpy as np
from transformers import BertTokenizerFast, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

# 3. Data Loading Function (ADJUSTED FOR YOUR DATA)
def load_and_label_data(json_path):
    with open(json_path) as f:
        data = json.load(f)

    cases = []
    for case in data:
        # Combine all text parts that exist
        text_parts = []
        for col in ['title', 'facts', 'arguments', 'judgment']:
            if col in case and case[col] and str(case[col]).strip():
                text_parts.append(str(case[col]).strip())

        if not text_parts:
            continue

        full_text = '\n\n'.join(text_parts)
        judgment_text = case.get('judgment', '').lower()

        # SIMPLIFIED LABELING (adjust these keywords as needed)
        if any(phrase in judgment_text for phrase in ['allowed', 'granted', 'approved']):
            label = 1
        elif any(phrase in judgment_text for phrase in ['dismissed', 'rejected', 'denied']):
            label = 0
        else:
            label = 0  # Default to 0 if unclear

        cases.append({
            'text': full_text,
            'label': label
        })

    return pd.DataFrame(cases)

# 4. Load and verify your data
try:
    df = load_and_label_data("/content/drive/MyDrive/processed_json/dataset.json")
    print(f"✅ Successfully loaded {len(df)} cases")
    print("\nLabel distribution:")
    print(df['label'].value_counts())

    # Show samples
    print("\nSample cases:")
    for i in range(min(3, len(df))):
        print(f"\nLabel: {df.iloc[i]['label']}")
        print(f"Text: {df.iloc[i]['text'][:200]}...")
except Exception as e:
    print(f"❌ Error loading data: {e}")
    exit()

# 5. Tokenization
tokenizer = BertTokenizerFast.from_pretrained("nlpaueb/legal-bert-base-uncased")

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

dataset = Dataset.from_pandas(df)
dataset = dataset.map(tokenize_function, batched=True)

# 6. Model Setup
model = BertForSequenceClassification.from_pretrained(
    "nlpaueb/legal-bert-base-uncased",
    num_labels=2
)

# 7. SIMPLIFIED Training Configuration (removed problematic arguments)
training_args = TrainingArguments(
    output_dir="./legalbert_results",
    per_device_train_batch_size=4,
    num_train_epochs=3,
    save_total_limit=2,
    learning_rate=2e-5,
    weight_decay=0.01
)

# 8. Trainer Setup
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer
)

# 9. Training
print("\n🚀 Starting training...")
trainer.train()
trainer.save_model("./legalbert_finetuned")
print("🎉 Training completed successfully!")

✅ Successfully loaded 30 cases

Label distribution:
label
1    30
Name: count, dtype: int64

Sample cases:

Label: 1
Text: 1

, and since the respondent did not raise any such Eighth
Amendment issue, Justice Powell concurred with the majority.
32.
The dissenting opinion of four Justices makes interesting
reading. Justice ...

Label: 1
Text: REPORTABLE

of
every case.”
5.
Thereafter, vide the order dated 29.06.2016, another bench of two
judges of this Court, on examining the questions formulated in T.P.
(C) No. 1118 of 2014, referred to A...

Label: 1
Text: 2023 INSC 1066

in Hyundai Engg. case [United India
Insurance Co. Ltd. v. Hyundai Engg. & Construction Co. Ltd.,
(2018) 17 SCC 607 : (2019) 2 SCC (Civ) 530] is important in
that what was specifically ...


Map:   0%|          | 0/30 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at nlpaueb/legal-bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-12-e4cceb101844>:93: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.



🚀 Starting training...


wandb: Currently logged in as: ditivasisht (ditivasisht-delhi-technical-campus) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss


🎉 Training completed successfully!


In [ ]:
print("\nSample positive cases:")
print(df[df['label'] == 1]['judgment_text'].head(3))

print("\nSample negative cases:")
print(df[df['label'] == 0]['judgment_text'].head(3))


Sample positive cases:
0    s and certain International Covenants,\nopined...
5     and records.\nPersonal Interview:\nThere shal...
7    s in ADR and PUCL\n55\nc.\nThe focal point of ...
Name: judgment_text, dtype: object

Sample negative cases:
1    s of this Court on the\napplicability of the p...
2     in Hyundai Engg. case [United India\nInsuranc...
3     of this Court in Azeez\nBasha (supra).  They ...
Name: judgment_text, dtype: object


In [ ]:
import pdfplumber
import torch
import re
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from transformers import BertTokenizerFast, BertForSequenceClassification
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import requests
from bs4 import BeautifulSoup
import json
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
import base64
from io import BytesIO

# 1. Load models
def load_models(model_path="./legalbert_finetuned"):
    """Load all necessary models for document analysis"""
    try:
        tokenizer = BertTokenizerFast.from_pretrained(model_path)
        classifier = BertForSequenceClassification.from_pretrained(model_path)
    except Exception as e:
        print(f"Error loading fine-tuned model: {e}")
        print("Using default BERT model as fallback")
        tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')
        classifier = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

    # Sentence embedder for semantic search
    embedder = SentenceTransformer('all-MiniLM-L6-v2')

    return tokenizer, classifier, embedder

# 2. Dataset Loading
def load_legal_dataset(file_path=None):
    """Load the legal cases dataset used for training the model"""
    dataset = []

    try:
        if file_path and os.path.exists(file_path):
            # Load from local CSV/Excel file
            if file_path.endswith('.csv'):
                df = pd.read_csv(file_path)
            elif file_path.endswith(('.xlsx', '.xls')):
                df = pd.read_excel(file_path)
            else:
                raise ValueError("Unsupported file format")

            # Extract cases with their outcomes and text
            for _, row in df.iterrows():
                # Adjust these field names based on your actual dataset structure
                case_name = row.get('case_name', '')
                outcome = row.get('outcome', '')
                text = row.get('text', '')
                provisions = row.get('provisions', '')

                if case_name and text:
                    dataset.append({
                        'case_name': case_name,
                        'outcome': outcome,
                        'text': text,
                        'provisions': provisions if isinstance(provisions, str) else ''
                    })
        else:
            # Fallback to a small default dataset
            print("Dataset file not found. Using a minimal fallback dataset.")
            dataset = [
                {"case_name": "State of MP vs Rameshwar", "outcome": "Dismissed",
                 "text": "Case regarding quashing power under Section 482", "provisions": "Section 482"},
                {"case_name": "B.S. Joshi vs State of Haryana", "outcome": "Allowed",
                 "text": "Matrimonial dispute regarding settlement", "provisions": "Section 320"},
                {"case_name": "Nikhil Merchant vs CBI", "outcome": "Allowed",
                 "text": "Settlement-based quashing of FIR", "provisions": "Section 482"}
            ]
    except Exception as e:
        print(f"Error loading dataset: {e}")
        print("Using minimal fallback dataset")
        dataset = [
            {"case_name": "State of MP vs Rameshwar", "outcome": "Dismissed",
             "text": "Case regarding quashing power under Section 482", "provisions": "Section 482"},
            {"case_name": "B.S. Joshi vs State of Haryana", "outcome": "Allowed",
             "text": "Matrimonial dispute regarding settlement", "provisions": "Section 320"}
        ]

    return dataset

# 3. PDF Text Extraction
def extract_text_from_pdf(file_bytes):
    """Extract text content from PDF file"""
    text = ""
    try:
        with pdfplumber.open(io.BytesIO(file_bytes)) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text() or ""
                text += page_text + "\n"
    except Exception as e:
        print(f"Error extracting text from PDF: {e}")

    return text

# 4. Advanced Text Preprocessing and Feature Extraction
def clean_and_extract_fields(text):
    """Extract structured information from legal document text"""
    # Case name extraction with improved regex
    case_match = re.search(r"([A-Za-z0-9\s\.,]+)\s+v[s\.]*\s+([A-Za-z0-9\s\.,&]+)", text, re.IGNORECASE)
    if not case_match:
        # Try alternative patterns
        case_match = re.search(r"([A-Za-z0-9\s\.,]+)\s+and\s+([A-Za-z0-9\s\.,&]+)", text, re.IGNORECASE)

    case_name = f"{case_match.group(1).strip()} vs {case_match.group(2).strip()}" if case_match else "Not found"

    # Extract parties with multiple patterns
    petitioner_patterns = [
        r"(?:learned counsel for the petitioner.*?:|Petitioner['']s*:?)(.*?)(?=\n)",
        r"(?:petitioner|plaintiff|appellant)[:]\s*(.*?)(?=\n)",
        r"(?:appearing for (?:the )?petitioner.*?:)(.*?)(?=\n)"
    ]

    respondent_patterns = [
        r"(?:learned counsel for the respondent.*?:|Respondent['']s*:?)(.*?)(?=\n)",
        r"(?:respondent|defendant|opposite party)[:]\s*(.*?)(?=\n)",
        r"(?:appearing for (?:the )?respondent.*?:)(.*?)(?=\n)"
    ]

    petitioner = None
    for pattern in petitioner_patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        if matches:
            petitioner = matches[0].strip()
            break

    respondent = None
    for pattern in respondent_patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        if matches:
            respondent = matches[0].strip()
            break

    # Extract legal provisions with improved regex
    provisions = []
    provision_patterns = [
        r"(Section|S\.|Article)\s*\d+[A-Za-z]*(?:\(\d+\))?(?:\([a-z]\))?(?:\s+of\s+[A-Za-z\s]+)?",
        r"(Sections|Articles)\s*\d+[A-Za-z]*(?:\s*,\s*\d+[A-Za-z]*)*(?:\s+of\s+[A-Za-z\s]+)?"
    ]

    for pattern in provision_patterns:
        provisions.extend(re.findall(pattern, text, re.IGNORECASE))

    # Clean provisions list
    provisions = sorted(set([p.strip() for p in provisions if p.strip()]))

    # Extract key findings with semantic importance
    sentences = re.split(r'(?<=[.])\s+', text)
    findings = []
    important_keywords = ['held that', 'we find', 'it is observed', 'conclude', 'therefore',
                          'the court', 'judgment', 'decided', 'ruled', 'ordered', 'decreed']

    for sentence in sentences:
        sentence = sentence.strip()
        if len(sentence) > 30:  # Filter out very short sentences
            if any(kw in sentence.lower() for kw in important_keywords):
                findings.append(sentence)

    return {
        "case_name": case_name,
        "petitioner": petitioner or "Not specified",
        "respondent": respondent or "Not specified",
        "provisions": provisions,
        "key_findings": findings[:5],  # limit for brevity
        "full_text": text
    }

# 5. Improved Prediction Function
def predict_outcome(text, tokenizer, classifier):
    """Predict the outcome of a legal case with confidence score"""
    # Split text into chunks to handle long documents
    max_length = 512
    chunks = [text[i:i+max_length*4] for i in range(0, len(text), max_length*4)]

    all_probs = []

    for chunk in chunks[:5]:  # Process up to 5 chunks for efficiency
        inputs = tokenizer(chunk, return_tensors="pt", truncation=True, max_length=max_length)
        with torch.no_grad():
            outputs = classifier(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
        all_probs.append(probs)

    # Average probabilities across chunks
    if all_probs:
        avg_probs = torch.mean(torch.cat(all_probs), dim=0)
        confidence = torch.max(avg_probs).item()
        prediction = "Allowed" if torch.argmax(avg_probs) == 1 else "Dismissed"
        return prediction, confidence
    else:
        return "Unknown", 0.0

# 6. Enhanced Similar Case Finder
def find_similar_cases(input_text, dataset, embedder, top_n=5):
    """Find semantically similar cases from the dataset using embeddings"""
    if not dataset or not input_text:
        return []

    # Generate embeddings
    try:
        input_emb = embedder.encode(input_text[:5000], convert_to_tensor=True)  # Limit text length for embedding

        # Extract case texts for comparison
        case_texts = [case.get('text', '')[:5000] for case in dataset]  # Limit text length for embedding
        case_names = [case.get('case_name', 'Unknown Case') for case in dataset]

        # Generate corpus embeddings
        corpus_emb = embedder.encode(case_texts, convert_to_tensor=True)

        # Calculate similarities
        similarities = util.pytorch_cos_sim(input_emb, corpus_emb)[0]

        # Get top matches
        top_indices = torch.topk(similarities, k=min(top_n, len(similarities))).indices.tolist()

        results = []
        for i in top_indices:
            results.append({
                "case_name": case_names[i],
                "similarity": float(similarities[i]),
                "outcome": dataset[i].get('outcome', 'Unknown'),
                "provisions": dataset[i].get('provisions', '')
            })

        return results
    except Exception as e:
        print(f"Error finding similar cases: {e}")
        return []

# 7. Online Search for Similar Cases (Fallback)
def search_online_for_cases(query, max_results=3):
    """Search online for similar legal cases as a fallback"""
    try:
        search_url = f"https://www.indiankanoon.org/search/?formInput={query.replace(' ', '+')}"
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}

        response = requests.get(search_url, headers=headers, timeout=5)
        soup = BeautifulSoup(response.text, 'html.parser')

        results = []
        for result in soup.select('.result_title')[:max_results]:
            case_name = result.get_text().strip()
            case_link = result.find('a')['href']
            results.append({
                'case_name': case_name,
                'link': f"https://www.indiankanoon.org{case_link}"
            })

        return results
    except Exception as e:
        print(f"Error searching online cases: {e}")
        return []


# 8. Data Visualization Functions
def create_confidence_gauge(confidence):
    """Create a gauge chart for confidence visualization"""
    fig, ax = plt.subplots(figsize=(6, 3))

    # Create gauge
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect('equal')
    ax.axis('off')

    # Draw gauge background
    theta = np.linspace(0, np.pi, 100)
    x = 0.5 + 0.4 * np.cos(theta)
    y = 0.5 + 0.4 * np.sin(theta)
    ax.plot(x, y, color='grey', linewidth=8, alpha=0.3)

    # Draw confidence level
    theta_conf = np.linspace(0, np.pi * confidence, 100)
    x_conf = 0.5 + 0.4 * np.cos(theta_conf)
    y_conf = 0.5 + 0.4 * np.sin(theta_conf)
    color = 'green' if confidence > 0.7 else 'orange' if confidence > 0.4 else 'red'
    ax.plot(x_conf, y_conf, color=color, linewidth=8)

    # Add text
    ax.text(0.5, 0.25, f"{confidence:.1%}", ha='center', va='center', fontsize=18, fontweight='bold')
    ax.text(0.5, 0.15, "Confidence", ha='center', va='center', fontsize=12)

    # Convert plot to base64 for embedding in HTML
    buffer = BytesIO()
    plt.savefig(buffer, format='png', bbox_inches='tight')
    buffer.seek(0)
    img_str = base64.b64encode(buffer.read()).decode('utf-8')
    plt.close()

    return f"data:image/png;base64,{img_str}"

def create_similarity_chart(similar_cases):
    """Create a bar chart for case similarity visualization"""
    if not similar_cases:
        return ""

    fig, ax = plt.subplots(figsize=(8, 4))

    cases = [case["case_name"].split(' vs ')[0] for case in similar_cases]
    similarities = [case["similarity"] for case in similar_cases]
    outcomes = [case["outcome"] for case in similar_cases]

    # Create colors based on outcome
    colors = ['#5cb85c' if outcome == 'Allowed' else
              '#d9534f' if outcome == 'Dismissed' else
              '#5bc0de' for outcome in outcomes]

    # Create horizontal bar chart
    y_pos = np.arange(len(cases))
    ax.barh(y_pos, similarities, color=colors)

    # Add labels and titles
    ax.set_yticks(y_pos)
    ax.set_yticklabels([name[:15] + "..." if len(name) > 15 else name for name in cases])
    ax.set_xlabel('Similarity Score')
    ax.set_title('Similar Cases')

    # Add a grid
    ax.grid(axis='x', linestyle='--', alpha=0.7)

    # Convert plot to base64 for embedding in HTML
    buffer = BytesIO()
    plt.savefig(buffer, format='png', bbox_inches='tight')
    buffer.seek(0)
    img_str = base64.b64encode(buffer.read()).decode('utf-8')
    plt.close()

    return f"data:image/png;base64,{img_str}"

def create_provisions_chart(provisions_list, dataset):
    """Create a visualization for legal provisions comparison"""
    if not provisions_list or not dataset:
        return ""

    # Count provisions in dataset
    all_provisions = {}
    for case in dataset:
        case_provisions = case.get('provisions', '')
        if isinstance(case_provisions, str):
            for provision in re.findall(r'(Section|S\.|Article)\s*\d+[A-Za-z]*', case_provisions):
                all_provisions[provision] = all_provisions.get(provision, 0) + 1

    # Filter to relevant provisions (mentioned in current case + top frequent)
    relevant_provisions = {}
    for p in provisions_list:
        simplified_p = re.search(r'(Section|S\.|Article)\s*\d+[A-Za-z]*', p)
        if simplified_p:
            relevant_provisions[simplified_p.group(0)] = all_provisions.get(simplified_p.group(0), 1)

    # If we have too few provisions, add some common ones from the dataset
    if len(relevant_provisions) < 3:
        top_provisions = sorted(all_provisions.items(), key=lambda x: x[1], reverse=True)[:5]
        for p, count in top_provisions:
            if p not in relevant_provisions:
                relevant_provisions[p] = count

    # Create plot
    fig, ax = plt.subplots(figsize=(8, 4))

    provisions = list(relevant_provisions.keys())
    counts = list(relevant_provisions.values())

    # Calculate outcome distribution for each provision
    outcome_allowed = []
    outcome_dismissed = []

    for provision in provisions:
        allowed = 0
        dismissed = 0
        for case in dataset:
            if provision in case.get('provisions', ''):
                if case.get('outcome') == 'Allowed':
                    allowed += 1
                elif case.get('outcome') == 'Dismissed':
                    dismissed += 1

        total = allowed + dismissed
        if total > 0:
            outcome_allowed.append(allowed / total)
            outcome_dismissed.append(dismissed / total)
        else:
            outcome_allowed.append(0)
            outcome_dismissed.append(0)

    # Create bar chart with outcome distribution
    x = np.arange(len(provisions))
    width = 0.35

    ax.bar(x, outcome_allowed, width, label='Allowed', color='#5cb85c')
    ax.bar(x, outcome_dismissed, width, bottom=outcome_allowed, label='Dismissed', color='#d9534f')

    # Add labels and title
    ax.set_ylabel('Proportion of Cases')
    ax.set_title('Outcome Distribution by Legal Provision')
    ax.set_xticks(x)
    ax.set_xticklabels([p[:10] + '...' if len(p) > 10 else p for p in provisions], rotation=45, ha='right')
    ax.legend()

    plt.tight_layout()

    # Convert plot to base64 for embedding in HTML
    buffer = BytesIO()
    plt.savefig(buffer, format='png', bbox_inches='tight')
    buffer.seek(0)
    img_str = base64.b64encode(buffer.read()).decode('utf-8')
    plt.close()

    return f"data:image/png;base64,{img_str}"

# 9. Main Analysis Function
def analyze_legal_document(file_bytes, dataset_path=None):
    """Complete analysis workflow for a legal document"""
    # Load all required components
    tokenizer, classifier, embedder = load_models()
    dataset = load_legal_dataset(dataset_path)

    # Extract and process text
    text = extract_text_from_pdf(file_bytes)
    if not text:
        return {"error": "Could not extract text from the document"}

    # Extract structured information
    data = clean_and_extract_fields(text)

    # Predict outcome
    prediction, confidence = predict_outcome(data['full_text'], tokenizer, classifier)

    # Find similar cases
    similar_cases = find_similar_cases(data['full_text'], dataset, embedder)

    # If no similar cases found in dataset, try online search as fallback
    if not similar_cases and data['case_name'] != 'Not found':
        try:
            search_query = f"{data['case_name']} {' '.join(data['provisions'][:3])}"
            similar_cases = search_online_for_cases(search_query)
        except Exception as e:
            print(f"Online search fallback failed: {e}")

    # Create visualizations
    confidence_gauge = create_confidence_gauge(confidence)
    similarity_chart = create_similarity_chart(similar_cases)
    provisions_chart = create_provisions_chart(data['provisions'], dataset)

    # Prepare result dict
    result = {
        "case_details": data,
        "prediction": {
            "outcome": prediction,
            "confidence": confidence
        },
        "similar_cases": similar_cases,
        "visualizations": {
            "confidence_gauge": confidence_gauge,
            "similarity_chart": similarity_chart,
            "provisions_chart": provisions_chart
        }
    }

    return result

# 10. UI Components
def create_ui_output(result):
    """Create beautiful HTML output for the analysis result"""
    case_details = result["case_details"]
    prediction = result["prediction"]
    similar_cases = result["similar_cases"]
    visualizations = result["visualizations"]

    # Build HTML
    html = f"""
    <style>
        .report {{
            font-family: 'Segoe UI', Arial, sans-serif;
            background: #1e1e1e;
            color: #e0e0e0;
            border-radius: 8px;
            padding: 25px;
            max-width: 1000px;
            margin: 0 auto;
        }}
        .header {{
            text-align: center;
            margin-bottom: 25px;
            border-bottom: 2px solid #444;
            padding-bottom: 15px;
        }}
        .section {{
            margin-bottom: 25px;
            padding: 15px;
            background: #282828;
            border-radius: 5px;
        }}
        .section-title {{
            font-weight: bold;
            border-bottom: 1px solid #444;
            color: #29b6f6;
            padding-bottom: 8px;
            margin-bottom: 15px;
        }}
        .flex-container {{
            display: flex;
            flex-wrap: wrap;
            gap: 20px;
        }}
        .flex-item {{
            flex: 1;
            min-width: 300px;
        }}
        .prediction-badge {{
            display: inline-block;
            padding: 5px 10px;
            border-radius: 4px;
            font-weight: bold;
            color: white;
            background-color: {{'#5cb85c' if prediction['outcome'] == 'Allowed' else '#d9534f'}};
        }}
        .similarity-item {{
            margin-bottom: 12px;
            padding: 10px;
            background: #333;
            border-radius: 4px;
            border-left: 4px solid {{'#5cb85c' if s.get('outcome') == 'Allowed' else '#d9534f' if s.get('outcome') == 'Dismissed' else '#5bc0de'}};
        }}
        .charts {{
            text-align: center;
            margin-top: 20px;
        }}
        .chart-container {{
            margin-bottom: 20px;
        }}
        .key-finding {{
            background: #333;
            padding: 12px;
            margin-bottom: 10px;
            border-radius: 4px;
            border-left: 4px solid #29b6f6;
        }}
        ul {{
            padding-left: 20px;
        }}
        li {{
            margin-bottom: 8px;
        }}
    </style>

    <div class="report">
        <div class="header">
            <h1>Legal Document Analysis Report</h1>
            <p>Case analysis powered by LegalBERT model with advanced similarity matching</p>
        </div>

        <div class="section">
            <h2 class="section-title">Case Details</h2>
            <p><strong>Case Name:</strong> {case_details['case_name']}</p>

            <div class="flex-container">
                <div class="flex-item">
                    <h3>Parties</h3>
                    <ul>
                        <li><strong>Petitioner:</strong> {case_details['petitioner']}</li>
                        <li><strong>Respondent:</strong> {case_details['respondent']}</li>
                    </ul>
                </div>
                <div class="flex-item">
                    <h3>Legal Provisions</h3>
                    <ul>
                        {'<li>No specific provisions identified</li>' if not case_details['provisions'] else ''.join(f"<li>{p}</li>" for p in case_details['provisions'][:5])}
                    </ul>
                </div>
            </div>
        </div>

        <div class="section">
            <h2 class="section-title">Prediction</h2>
            <div class="flex-container">
                <div class="flex-item">
                    <p><strong>Outcome:</strong> <span class="prediction-badge">{prediction['outcome']}</span></p>
                    <p><strong>Confidence:</strong> {prediction['confidence']:.2%}</p>
                </div>
                <div class="flex-item charts">
                    <img src="{visualizations['confidence_gauge']}" style="max-width: 300px;">
                </div>
            </div>
        </div>

        <div class="section">
            <h2 class="section-title">Key Findings</h2>
            {'<p>No key findings extracted</p>' if not case_details['key_findings'] else ''.join(f'<div class="key-finding">{finding}</div>' for finding in case_details['key_findings'])}
        </div>

        <div class="section">
            <h2 class="section-title">Similar Cases</h2>

            <div class="flex-container">
                <div class="flex-item">


                </div>
                <div class="flex-item charts">
                    <div class="chart-container">
                        <img src="{visualizations['similarity_chart']}" style="max-width: 100%;">
                    </div>
                </div>
            </div>
        </div>

        <div class="section">
            <h2 class="section-title">Legal Provisions Analysis</h2>
            <div class="charts">
                <img src="{visualizations['provisions_chart']}" style="max-width: 100%;">
                <p>This chart shows outcome distribution for cases with similar legal provisions</p>
            </div>
        </div>
    </div>
    """

    return html

# 11. Demo Usage Function for Jupyter Notebook
def setup_ui_components(dataset_path=None):
    """Set up interactive UI components for document analysis"""
    # Create widgets
    upload_btn = widgets.FileUpload(accept='.pdf', multiple=False, description='Upload Legal PDF')
    dataset_upload = widgets.FileUpload(accept='.csv,.xlsx', multiple=False, description='Upload Dataset (Optional)')
    analyze_btn = widgets.Button(description='Analyze Document')
    output_area = widgets.Output()

    # Define callbacks
    def on_analyze_click(b):
        with output_area:
            clear_output()

            if not upload_btn.value:
                print("⚠️ Please upload a PDF document first.")
                return

            # Get PDF content
            uploaded_pdf = list(upload_btn.value.values())[0]['content']

            # Check for dataset upload
            dataset_file = None
            if dataset_upload.value:
                try:
                    dataset_content = list(dataset_upload.value.values())[0]['content']
                    dataset_name = list(dataset_upload.value.values())[0]['name']

                    # Save dataset temporarily
                    dataset_file = f"/tmp/{dataset_name}"
                    with open(dataset_file, "wb") as f:
                        f.write(dataset_content)

                    print(f"Using uploaded dataset: {dataset_name}")
                except Exception as e:
                    print(f"Error processing dataset: {e}")
                    dataset_file = None

            # Analyze document
            try:
                print("⏳ Analyzing document... please wait.")
                result = analyze_legal_document(uploaded_pdf, dataset_path=dataset_file or dataset_path)
                display(HTML(create_ui_output(result)))
            except Exception as e:
                print(f"⚠️ Error during analysis: {e}")
                import traceback
                traceback.print_exc()

    # Connect callbacks
    analyze_btn.on_click(on_analyze_click)

    # Display UI
    display(widgets.VBox([
        widgets.HBox([upload_btn, dataset_upload]),
        analyze_btn,
        output_area
    ]))

    return {
        "upload_btn": upload_btn,
        "dataset_upload": dataset_upload,
        "analyze_btn": analyze_btn,
        "output_area": output_area
    }

# Example usage
if __name__ == "__main__":
    # This would be in a Jupyter notebook
    setup_ui_components(dataset_path="legal_cases_dataset.csv")